# Exercices de la Rencontre 12 : NumPy (avancé) et IA (bases)

Ce notebook regroupe les exercices de la rencontre 12. Vous y pratiquerez :

- le *broadcasting* et les opérations vectorisées avancées (normalisation, corrélation) ;
- les masques booléens multiples et `np.where()` ;
- la restructuration de tableaux (`reshape`, `transpose`, `vstack`, `hstack`, `concatenate`) et la sauvegarde `.npy` ;
- les bases du Machine Learning avec scikit-learn : régression et classification.

Essayez d'abord chaque exercice par vous-même avant de consulter la solution proposée.

### Accès aux jeux de données

Exécutez la cellule suivante en premier. La fonction `chemin_donnees()` cherche d'abord le fichier CSV dans le même dossier que le notebook (comme demandé dans les exercices du cours). Si le fichier n'y est pas (par exemple dans Google Colab), elle utilise la copie en ligne du dépôt du cours.

In [ ]:
import os

URL_DONNEES = "https://raw.githubusercontent.com/Simonll/420-SN1/develop/web/static/ressources/csv/"


def chemin_donnees(nom_fichier):
    """Retourne le chemin local du fichier s'il existe, sinon son adresse en ligne."""
    for dossier in [".", "../../web/static/ressources/csv"]:
        chemin = os.path.join(dossier, nom_fichier)
        if os.path.exists(chemin):
            return chemin
    return URL_DONNEES + nom_fichier


print(chemin_donnees("tips_dataset.csv"))

## Exercices de création

### Exercice 1 - Broadcasting

1. Créez un tableau 2D de forme (3, 4) contenant des entiers aléatoires entre 0 et 10 ;
2. Ajoutez un vecteur ligne de 4 éléments à chaque ligne du tableau ;
3. Multipliez chaque **ligne** par un scalaire différent à l'aide d'un vecteur colonne de forme (3, 1) ;
4. Affichez les formes et le résultat final.

**Rappel :** NumPy « étire » automatiquement un tableau de taille 1 dans une dimension pour qu'il corresponde à l'autre tableau.

In [ ]:
import numpy as np

np.random.seed(12)
tableau = np.random.randint(0, 11, size=(3, 4))
print("Tableau de départ (3, 4) :")
print(tableau)

ligne = np.array([100, 200, 300, 400])        # forme (4,)
avec_ligne = tableau + ligne
print("\n+ vecteur ligne", ligne.shape, ":")
print(avec_ligne)

colonne = np.array([[1], [10], [100]])        # forme (3, 1)
resultat = avec_ligne * colonne
print("\n* vecteur colonne", colonne.shape, ":")
print(resultat)
print("Forme finale :", resultat.shape)

### Exercice 2 - Opérations vectorisées avancées

1. Chargez le fichier `mpg_dataset.csv` avec Pandas ;
2. Extrayez les colonnes `mpg`, `horsepower` et `weight` en tableaux NumPy (attention : `horsepower` contient des valeurs manquantes à retirer d'abord) ;
3. Normalisez chaque colonne : soustrayez la moyenne, puis divisez par l'écart-type ;
4. Calculez la matrice de corrélation entre ces 3 colonnes avec `np.corrcoef` ;
5. Affichez la matrice de corrélation et interprétez-la.

In [ ]:
import pandas as pd

df_mpg = pd.read_csv(chemin_donnees("mpg_dataset.csv"))
df_mpg = df_mpg.dropna(subset=["mpg", "horsepower", "weight"])

donnees = df_mpg[["mpg", "horsepower", "weight"]].to_numpy()   # forme (n, 3)
print("Forme :", donnees.shape)

# Broadcasting : la moyenne et l'écart-type (forme (3,)) s'appliquent à chaque ligne
normalisees = (donnees - donnees.mean(axis=0)) / donnees.std(axis=0)
print("Moyennes après normalisation    :", normalisees.mean(axis=0).round(3))
print("Écarts-types après normalisation :", normalisees.std(axis=0).round(3))

correlation = np.corrcoef(normalisees, rowvar=False)
print("\nMatrice de corrélation (mpg, horsepower, weight) :")
print(correlation.round(3))

# Interprétation : mpg est fortement et négativement corrélé à horsepower et à weight
# (une voiture plus lourde ou plus puissante parcourt moins de milles par gallon).

### Exercice 3 - Filtres multiples avec NumPy

Utilisez le fichier `tips_dataset.csv`.

1. Chargez les données avec Pandas ;
2. Extrayez les colonnes `total_bill`, `tip` et `size` en tableaux NumPy ;
3. Créez un masque pour les repas où `total_bill > 20` **ET** `tip > 3` **ET** `size >= 2` ;
4. Appliquez le masque et affichez les données filtrées ;
5. Comptez le nombre de repas correspondant aux critères ;
6. Créez un second masque pour les repas où `total_bill < 10` **OU** `tip < 2` ;
7. Avec `np.where()`, créez un tableau d'étiquettes « généreux » / « standard » selon que le pourboire dépasse 15 % de la facture.

In [ ]:
df_tips = pd.read_csv(chemin_donnees("tips_dataset.csv"))
total_bill = df_tips["total_bill"].to_numpy()
tip = df_tips["tip"].to_numpy()
size = df_tips["size"].to_numpy()

masque_et = (total_bill > 20) & (tip > 3) & (size >= 2)
# Équivalent : np.logical_and(np.logical_and(total_bill > 20, tip > 3), size >= 2)
print("Factures filtrées :", total_bill[masque_et][:10], "...")
print("Pourboires filtrés :", tip[masque_et][:10], "...")
print("Nombre de repas (ET) :", masque_et.sum())

masque_ou = np.logical_or(total_bill < 10, tip < 2)
print("Nombre de repas (OU) :", masque_ou.sum())

etiquettes = np.where(tip / total_bill > 0.15, "généreux", "standard")
valeurs, comptes = np.unique(etiquettes, return_counts=True)
print("\nRépartition :", dict(zip(valeurs.tolist(), comptes.tolist())))

### Exercice 4 - Restructurer des données

1. Générez un tableau 1D de 24 éléments ;
2. Restructurez-le en forme (2, 3, 4) : 2 matrices de 3 lignes et 4 colonnes ;
3. Extrayez chaque matrice 2D et calculez la somme de chacune ;
4. Transposez les deux matrices avec `np.transpose` ;
5. Concaténez les deux matrices transposées avec `np.concatenate` ;
6. Comparez avec `np.vstack()` et `np.hstack()` ;
7. Sauvegardez le résultat dans un fichier `.npy` et rechargez-le pour vérifier.

In [ ]:
valeurs = np.arange(24)
cube = valeurs.reshape(2, 3, 4)
print("Forme du cube :", cube.shape)

matrice_a = cube[0]
matrice_b = cube[1]
print("Somme de la matrice A :", matrice_a.sum())
print("Somme de la matrice B :", matrice_b.sum())

a_t = np.transpose(matrice_a)   # (4, 3)
b_t = np.transpose(matrice_b)   # (4, 3)

concatene = np.concatenate([a_t, b_t])    # par défaut, axis=0
print("\nconcatenate :", concatene.shape)
print("vstack      :", np.vstack([a_t, b_t]).shape)
print("hstack      :", np.hstack([a_t, b_t]).shape)

np.save("resultat.npy", concatene)
recharge = np.load("resultat.npy")
print("\nRechargé identique à l'original :", np.array_equal(concatene, recharge))

### Exercice 5 - Introduction à l'IA (réflexion)

Répondez brièvement dans la cellule markdown ci-dessous (double-cliquez pour la modifier) :

1. Qu'est-ce que l'Intelligence Artificielle (IA) ?
2. Quelle est la différence entre IA faible et IA forte ?
3. Qu'est-ce que le Machine Learning (ML) ?
4. Quelles sont les principales catégories de ML ?
5. Donnez un exemple concret d'application du ML dans un domaine scientifique.

*Vos réponses :*

1. ...
2. ...
3. ...
4. ...
5. ...

### Exercice 6 - Régression linéaire avec scikit-learn

1. Utilisez `make_regression` de scikit-learn pour générer 100 points avec une seule variable explicative et un peu de bruit ;
2. Entraînez un modèle `LinearRegression` ;
3. Évaluez le modèle avec le R² sur les données d'entraînement (`model.score`) ;
4. Tracez les points et la droite de régression.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

X, y = make_regression(n_samples=100, n_features=1, noise=15, random_state=42)

model = LinearRegression()
model.fit(X, y)

print(f"Pente     : {model.coef_[0]:.2f}")
print(f"Intercept : {model.intercept_:.2f}")
print(f"R² (entraînement) : {model.score(X, y):.3f}")

x_droite = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.scatter(X, y, alpha=0.6, label="Données")
plt.plot(x_droite, model.predict(x_droite), color="crimson", label="Droite de régression")
plt.title("Régression linéaire simple")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercice 7 - Machine Learning complet avec scikit-learn (classification)

On veut prédire si un étudiant réussira (1) ou non (0) à partir de ses heures d'étude et de sa moyenne.

1. Créez un jeu de données `X` (2 colonnes : heures d'étude, moyenne sur 100) et `y` (0 ou 1) pour une dizaine d'étudiants ;
2. **Normalisez** les données avec `StandardScaler` (`fit_transform`) ;
3. Séparez en entraînement / test avec `train_test_split(X, y, test_size=0.3, random_state=42)` ;
4. Entraînez un `LogisticRegression` avec `.fit(X_train, y_train)` ;
5. Prédisez avec `.predict(X_test)` et calculez le score avec `accuracy_score(y_test, y_pred)` ;
6. Affichez la précision (*accuracy*) en pourcentage.

Résultat attendu (exemple) :

```text
Précision du modèle : 100.0 %
```

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = np.array([
    [2, 55], [3, 60], [1, 50], [4, 62], [5, 70],
    [8, 78], [10, 85], [7, 80], [9, 88], [12, 92],
])
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

scaler = StandardScaler()
X_normalise = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_normalise, y, test_size=0.3, random_state=42)

modele = LogisticRegression()
modele.fit(X_train, y_train)

y_pred = modele.predict(X_test)
precision = accuracy_score(y_test, y_pred)
print("Précision du modèle :", round(precision * 100, 1), "%")

# Prédiction pour un nouvel étudiant : 6 heures d'étude, moyenne de 72
nouveau = scaler.transform([[6, 72]])
print("Prédiction pour (6 h, 72) :", "réussite" if modele.predict(nouveau)[0] == 1 else "échec")

## Poursuite du TP2

Utilisez au moins une technique NumPy avancée (broadcasting, filtres multiples, `reshape`) dans votre TP2. Si votre analyse s'y prête, essayez une régression ou une classification simple, et documentez dans votre rapport ce que ces techniques ont apporté.